# 通过谐振拟合确定介电常数和损耗

在这个示例中，我们将从测量得到的散射参数数据中拟合出 Q 因子，用于表征 PCB 覆层材料。这些数据分别对应于 36 毫米、72 毫米和 144 毫米长的带状传输线谐振器。然后，我们将推导出介电常数和损耗常数。

## 拟合 Q 值

In [ ]:
import matplotlib.pyplot as plt

import skrf as rf
from skrf.constants import c
from skrf.qfactor import Qfactor

rf.stylely()

In [ ]:
# Loading the data.
# Setting the frequency unit is optional, just a convenience for displaying results
Reson36 = rf.Network('data/resonator_36mm.s2p', f_unit='GHz')
Reson72 = rf.Network('data/resonator_72mm.s2p', f_unit='GHz')
Reson144 = rf.Network('data/resonator_144mm.s2p', f_unit='GHz')

In [ ]:
Reson36.plot_s_db(m=1, n=0)
Reson72.plot_s_db(m=1, n=0)
Reson144.plot_s_db(m=1, n=0)

为了确定这些谐振器的 Q 值，我们从想要拟合的传输散射参数中创建一个 `Qfactor` 对象。请注意，如果存在许多彼此靠近的谐振峰，建议传递感兴趣频率范围内的频率子网络，以帮助拟合算法的收敛。在这里，我们关注的是发生在 2 GHz 和 4 GHz 附近的谐振峰，因此：

In [ ]:
# Around 2 GHz
QF_36_2GHz = Qfactor(Reson36['1.75-2.25GHz'].s21, res_type='transmission')
QF_72_2GHz = Qfactor(Reson72['1.75-2.25GHz'].s21, res_type='transmission')
QF_144_2GHz = Qfactor(Reson144['1.75-2.25GHz'].s21, res_type='transmission')
# Around 4 GHz
QF_36_4GHz = Qfactor(Reson36['3.75-4.25GHz'].s21, res_type='transmission')
QF_72_4GHz = Qfactor(Reson72['3.75-4.25GHz'].s21, res_type='transmission')
QF_144_4GHz = Qfactor(Reson144['3.75-4.25GHz'].s21, res_type='transmission')

all_QFs = [QF_36_2GHz, QF_72_2GHz, QF_144_2GHz, QF_36_4GHz, QF_72_4GHz, QF_144_4GHz]

In [ ]:
print(*all_QFs, sep='\n')

然后，我们拟合 Q 值：

In [ ]:
[Q.fit() for Q in all_QFs]
print(*all_QFs, sep='\n')

我们可以根据拟合结果创建相应的网络模型，并将其与测量结果进行对比，以评估模型的准确性：

In [ ]:
new_freq = rf.Frequency(1, 5, unit='GHz', npoints=1001)
fitted_ntwk_36_2GHz = QF_36_2GHz.fitted_network(frequency=new_freq)
fitted_ntwk_72_2GHz = QF_72_2GHz.fitted_network(frequency=new_freq)
fitted_ntwk_144_2GHz = QF_144_2GHz.fitted_network(frequency=new_freq)

fitted_ntwk_36_4GHz = QF_36_4GHz.fitted_network(frequency=new_freq)
fitted_ntwk_72_4GHz = QF_72_4GHz.fitted_network(frequency=new_freq)
fitted_ntwk_144_4GHz = QF_144_4GHz.fitted_network(frequency=new_freq)

In [ ]:
Reson36.plot_s_db(m=1, n=0, color='C0')
fitted_ntwk_36_2GHz.plot_s_db(label='Fitted Model ~ 2GHz', lw=2, color='C0', ls='--')
fitted_ntwk_36_4GHz.plot_s_db(label='Fitted Model ~ 4GHz', lw=2, color='C0', ls=':')
Reson72.plot_s_db(m=1, n=0, color='C1')
fitted_ntwk_72_2GHz.plot_s_db(label='Fitted Model ~ 2GHz', lw=2, color='C1', ls='--')
fitted_ntwk_72_4GHz.plot_s_db(label='Fitted Model ~ 4GHz', lw=2, color='C1', ls=':')
Reson144.plot_s_db(m=1, n=0, color='C2')
fitted_ntwk_144_2GHz.plot_s_db(label='Fitted Model ~ 2GHz', lw=2, color='C2', ls='--')
fitted_ntwk_144_4GHz.plot_s_db(label='Fitted Model ~ 4GHz', lw=2, color='C2', ls=':')
plt.gca().legend(ncol=3)

另一种表示结果的方法是使用极坐标图：

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
Reson36.plot_s_polar(m=1, n=0, ax=ax, ls='', marker='x', ms=5)
fitted_ntwk_36_2GHz.plot_s_polar(ax=ax, label="Fitted Model ~ 2GHz", lw=2, ls='--', color='C3')
fitted_ntwk_36_4GHz.plot_s_polar(ax=ax, label="Fitted Model ~ 4GHz", lw=2, ls=':', color='C3')

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
Reson72.plot_s_polar(m=1, n=0, ax=ax, ls='', marker='x', ms=5, color='C1')
fitted_ntwk_72_2GHz.plot_s_polar(ax=ax, label="Fitted Model ~ 2GHz", lw=2, ls='--', color='C3')
fitted_ntwk_72_4GHz.plot_s_polar(ax=ax, label="Fitted Model ~ 4GHz", lw=2, ls=':', color='C3')

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
Reson144.plot_s_polar(m=1, n=0, ax=ax, ls='', marker='x', ms=5, color='C2')
fitted_ntwk_144_2GHz.plot_s_polar(ax=ax, label="Fitted Model ~ 2GHz", lw=2, ls='--', color='C3')
fitted_ntwk_144_4GHz.plot_s_polar(ax=ax, label="Fitted Model ~ 4GHz", lw=2, ls=':', color='C3')

## 提高精度如果您希望在计算 Q 值时获得最佳精度，建议重复进行拟合，使用具有中心频率共振峰且扫描范围缩减至 $\pm f_L/Q_L$ 的 S 参数数据（参见 [1] 第 21 页）。这样做有以下两个理由：* $Q_L$ 是在谐振频率 $f_L$ 处定义的。* 限制扫描范围可以减少其他模式、阻抗失配以及实际谐振器与用于谐振器的集总元件模型之间的差异所产生的影响。此外，它还在峰值附近增加了更多数据点。如果谐振曲线形状较差，或者附近存在其他模式，则这种方法的优势最为明显。因此，为了获得最佳精度，建议的流程如下：1. 设置扫描参数，使谐振峰位于 VNA 屏幕上。2. 获取测量数据。3. 拟合 Q 参数，以推算出 $f_L$。4. 将中心频率设置为 $f_L$。5. 将扫描范围设置为 $2f_L/Q_L$。6. 获取测量数据。7. 拟合 Q 参数。8. 计算无负载 Q 值。例如：

In [ ]:
# extracting the subfrequency range of interest from the data
f_L = QF_72_2GHz.f_L
Q_L = QF_72_2GHz.Q_L
span = f_L/Q_L
ntwk = Reson72[f'{f_L - span/2}-{f_L + span/2}Hz']

In [ ]:
QF_72_2GHz_2 = Qfactor(ntwk.s21, res_type='transmission')

In [ ]:
# After the second fit, the weighting ratio is close to 0.2 at the minimum and maximum frequencies
# provided that the loop_plan contains ‘w’. (cf. [1] Fig. 6(b) and equation (28))
QF_72_2GHz_2.fit()

In [ ]:
QF_72_2GHz_2.Q_L  # higher accury fit

## 推导介电常数和损耗在[2]中给出了从微带谐振器的测量数据中计算 Dk（也称为 $\epsilon_r$）和 Df（也称为 $\tan \delta$）的公式：$$\epsilon_r = \left[ \frac{c n}{2 f_r (L + \Delta L)} \right]^2$$其中：- $L$ 是谐振器的长度，- $n$ 是谐振器中谐振时的半波长的数量，- $f_r$ 是谐振器的谐振频率，- $c$ 是真空中的光速，- $\Delta L$ 是由于谐振带末端的边缘场引起的谐振带总有效长度的增加量（在下面的示例中忽略）。并且：$$ \tan \delta = \frac{1}{Q} - \frac{1}{Q_c} $$其中：- $Q_c$ 是与铜损耗相关的 Q 值：根据 IPC，在 2 GHz 时为 250，在 4 GHz 时为 360。

因此，对于介电常数：

def Dk(f, L, n):    "Calculates Dk from resonator data. Eq.(1) of ref [1]"    return (n*c/(2*f*L))**2

In [ ]:
def Dk(f, L, n):
    "Calculates Dk from resonator data. Eq.(1) of ref [1]"
    return (n*c/(2*f*L))**2

In [ ]:
Dk(QF_72_2GHz.f_L, 72e-3, 2)

In [ ]:
Dk(QF_72_2GHz.f_L, 144e-3, 4)

Qc = 250  # from IPC ref [1]print(1/QF_36_2GHz.Q_L - 1/Qc)

In [ ]:
Qc = 250  # from IPC ref [1]
print(1/QF_36_2GHz.Q_L - 1/Qc)

In [ ]:
Qc = 250  # from IPC ref [1]
print(1/QF_144_2GHz.Q_L - 1/Qc)

In [ ]:
Qc = 360  # from IPC
print(1/QF_36_4GHz.Q_L - 1/Qc)

In [ ]:
Qc = 360  # from IPC
print(1/QF_144_4GHz.Q_L - 1/Qc)

## References
- [1] A. P. Gregory, "Q-factor Measurement by using a Vector Network Analyser", National Physical Laboratory Report MAT 58 (2021), https://eprintspublications.npl.co.uk/9304/
- [2] IPC-TM-650 TEST METHODS MANUAL, https://www.ipc.org/sites/default/files/test_methods_docs/2-5_2-5-5-5.pdf